# 01 – Data Exploration
**Purpose:** Load, inspect and clip any geodataset to a study area (AOI).
SWORD is the primary dataset. Additional datasets can be added in Section 2.

**Workflow:**
1.0 Imports

1.1 Configuration

1.2 Load & inspect SWORD

1.3 Load & inspect additional datasets

1.4 Define AOI and clip all datasets

1.5 Save outputs

1.6 Interactive map

---
## 1.0 | Imports

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import fiona
import os
import sys
import webbrowser
import time
from tqdm import tqdm
sys.path.append(os.path.join("..", "src"))

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT
from _2_1_2_point import snap_gdw_to_sword_nodes
from _00_config import STUDY_AREA, PFAF_IDS, PFAF_LEVEL_DIGITS

---
## 1.1 | Configuration
**Edit only this section.** All other cells run without changes.

In [ ]:
# ============================================================
# 1.1 SWORD – primary dataset
# ============================================================
SWORD_PATH  = os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "as_sword_reaches_v17b.gpkg")
SWORD_LAYER = None   # set to layer name if GeoPackage has multiple layers, else None

IN_SWORD_NODES = os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "as_sword_nodes_v17b.gpkg")
# ============================================================
# 1.2 Additional datasets (add as many as needed)
# Each entry: (label, path, layer_or_None)
# ============================================================
ADDITIONAL_DATASETS = [
    # ("My second dataset", os.path.join(DATA_RAW, "path", "to", "file.gpkg"), None),
    # ("Geology",           os.path.join(DATA_RAW, "path", "to", "glim.gpkg"), None),
]

# ============================================================
# 1.3 Area of interest (AOI) – used to clip all datasets
# ============================================================
AOI_PATH        = os.path.join(DATA_RAW, "0_data", "Naryn_example", "Naryn_Basin.gpkg")
AOI_LAYER       = None   # set to layer name if multiple layers, else None
AOI_FILTER_COL  = None   # e.g. "basin_name" – filter by attribute, else None
AOI_FILTER_VAL  = None   # e.g. "Naryn"       – filter value, else None

# ============================================================
# 1.4 Outputs
# ============================================================
OUT_SWORD_CLIPPED = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_clip.gpkg")
OUT_MAP           = os.path.join(DATA_OUTPUT,    "01_explore_map.html")

# ============================================================
# GDW
# ============================================================
IN_GDW = os.path.join(DATA_RAW, "0_data", "Naryn_example", "naryn_gdw_barriers_v1_0.gpkg")
IN_GDW_SNAP = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_2_1_gdw_snapped.gpkg")

# ============================================================
# FFR
# ============================================================
IN_FFR = os.path.join(DATA_RAW, "0_data", "FFR", "FFR_river_network.gdb")


# ============================================================
print("Configuration set:")
print(f"  SWORD        : {SWORD_PATH}")
print(f"  SWORD NODES       : {IN_SWORD_NODES}")
print(f"  AOI          : {AOI_PATH}")
print(f"  Output clip  : {OUT_SWORD_CLIPPED}")
print(f"  Output map   : {OUT_MAP}")
print(f"  GDW barriers  : {IN_GDW}")

Verify all files exist

In [ ]:
check_paths = [("SWORD", SWORD_PATH), ("AOI", AOI_PATH), ("SWORD NODES", IN_SWORD_NODES), ("GDW-barriers", IN_GDW)]
check_paths += [(label, path) for label, path, _ in ADDITIONAL_DATASETS]

all_ok = True
for label, path in check_paths:
    exists = os.path.exists(path)
    status = "Found" if exists else "NOT FOUND, check path in Section 1 (Configuration)"
    print(f"  {label:25s}: {status}")
    if not exists:
        all_ok = False

print()
print("All files found" if all_ok else "Fix missing paths in Section 1 before continuing.")

---
## 1.2 | Load & inspect SWORD

In [ ]:
# Inspect available layers
layers = fiona.listlayers(SWORD_PATH)
print(f"Layers in SWORD GeoPackage: {layers}")

# Load
layer = SWORD_LAYER if SWORD_LAYER else layers[0]
sword = gpd.read_file(SWORD_PATH, layer=layer)

print(f"\nReaches loaded : {len(sword)}")
print(f"CRS            : {sword.crs}")
print(f"Geometry type  : {sword.geom_type.unique()}")
print(f"Bounding box   : {sword.total_bounds}")

# Column overview
print(f"Columns ({len(sword.columns)}):")
print(sword.columns.tolist())

Inspecting nodes to compare last digit of node_id with GDW point dataset.

In [ ]:
nodes = gpd.read_file(IN_SWORD_NODES)

# Filter to Naryn nodes (first 5 digits = 46219)
naryn_nodes = nodes[nodes["node_id"].astype(str).str[:5] == "46219"]
print(f"Total Naryn nodes: {len(naryn_nodes)}")

# Extract last digit of node_id (type identifier)
naryn_nodes = naryn_nodes.copy()
naryn_nodes["node_type"] = naryn_nodes["node_id"].astype(str).str[-1].astype(int)

# Count nodes with type=4 (dam or waterfall)
dam_nodes = naryn_nodes[naryn_nodes["node_type"] == 4]
print(f"Dam/waterfall nodes (type=4): {len(dam_nodes)}")
print(f"\nNode type distribution:")
print(naryn_nodes["node_type"].value_counts().sort_index())

In [ ]:
gdw = gpd.read_file(IN_GDW)

print(f"GDW points in Naryn: {len(gdw)}")
print(f"Columns: {gdw.columns.tolist()}")

In [ ]:
toktogul = gdw[gdw["DAM_NAME"] == "Toktogul"]
print(toktogul[["DAM_NAME", "geometry"]])

dam_nodes = naryn_nodes[naryn_nodes["obstr_type"] > 0].copy()
toktogul_m = toktogul.to_crs("EPSG:32643")
dam_nodes_m = dam_nodes.to_crs("EPSG:32643")

for idx, row in toktogul_m.iterrows():
    distances = dam_nodes_m.geometry.distance(row.geometry)
    print(f"\nDistances from Toktogul to dam nodes:")
    print(distances.sort_values())

In [ ]:
print(sword[["reach_id", "rch_id_up", "rch_id_dn", "n_rch_up", "n_rch_dn"]].head(10))
print(f"\nDatatype rch_id_up: {sword['rch_id_up'].dtype}")
print(f"Type of value: {type(sword['rch_id_up'].iloc[0])}")

In [ ]:
# Find a reach with multiple upstream neighbors
multi_up = sword[sword["n_rch_up"] > 1].iloc[0]
print(f"reach_id: {multi_up['reach_id']}")
print(f"rch_id_up: '{multi_up['rch_id_up']}'")
print(f"n_rch_up: {multi_up['n_rch_up']}")

# Try splitting on different separators
print(f"\nSplit on whitespace: {multi_up['rch_id_up'].split()}")
print(f"Split on comma: {multi_up['rch_id_up'].split(',')}")

In [ ]:
# import folium
# import matplotlib
# import matplotlib.colors as mcolors

# # Normalize dist_out to 0-1 for colormap
# dist_min = sword["dist_out"].min()
# dist_max = sword["dist_out"].max()
# cmap = matplotlib.colormaps.get_cmap("Blues")

# center = [naryn_nodes.geometry.y.mean(), naryn_nodes.geometry.x.mean()]
# m = folium.Map(location=center, zoom_start=8, tiles="CartoDB positron")

# # Add SWORD reaches – color by dist_out
# for _, row in sword.iterrows():
#     norm = (row["dist_out"] - dist_min) / (dist_max - dist_min)
#     color = mcolors.to_hex(cmap(norm))
#     folium.GeoJson(
#         row.geometry,
#         style_function=lambda x, c=color: {
#             "color": c,
#             "weight": 3,
#             "opacity": 0.8
#         },
#         tooltip=f"Reach: {row['reach_id']} | dist_out: {row['dist_out']:.0f}m"
#     ).add_to(m)

# # Add SWORD dam nodes (blue)
# for _, row in dam_nodes.iterrows():
#     folium.CircleMarker(
#         location=[row.geometry.y, row.geometry.x],
#         radius=8, color="blue", fill=True,
#         fill_color="blue", fill_opacity=0.9,
#         tooltip=f"SWORD dam: {row['node_id']}"
#     ).add_to(m)

# # Add GDW points (red)
# for _, row in gdw.iterrows():
#     folium.CircleMarker(
#         location=[row.geometry.y, row.geometry.x],
#         radius=8, color="red", fill=True,
#         fill_color="red", fill_opacity=0.9,
#         tooltip=f"GDW: {row.get('DAM_NAME', '-')}"
#     ).add_to(m)

# out_path = os.path.join(DATA_OUTPUT, "dam_comparison_map.html")
# m.save(out_path)
# print(f"File size: {os.path.getsize(out_path) / 1024:.1f} KB")
# webbrowser.open(out_path)

In [ ]:
# print(os.path.exists(out_path))
# print(out_path)

---
## 1.3 | Load & inspect additional datasets
Add datasets in Section 1.2. This cell loops through all of them automatically.

In [ ]:
# Dictionary to store all additional datasets: {label: GeoDataFrame}
extra_datasets = {}

if not ADDITIONAL_DATASETS:
    print("No additional datasets defined in Section 1.2 – skipping.")
else:
    for label, path, layer_name in ADDITIONAL_DATASETS:
        print(f"Loading: {label}")
        layers_available = fiona.listlayers(path)
        lyr = layer_name if layer_name else layers_available[0]
        gdf = gpd.read_file(path, layer=lyr)
        extra_datasets[label] = gdf

        print(f"  Features  : {len(gdf)}")
        print(f"  CRS       : {gdf.crs}")
        print(f"  Columns   : {gdf.columns.tolist()}")
        print()

---
## 1.4 | Define AOI and clip all datasets

In [ ]:
# Load AOI
aoi_layers = fiona.listlayers(AOI_PATH)
aoi_layer  = AOI_LAYER if AOI_LAYER else aoi_layers[0]
aoi = gpd.read_file(AOI_PATH, layer=aoi_layer)

# Optional attribute filter
if AOI_FILTER_COL and AOI_FILTER_VAL:
    aoi = aoi[aoi[AOI_FILTER_COL] == AOI_FILTER_VAL]
    print(f"AOI filtered: {AOI_FILTER_COL} == {AOI_FILTER_VAL}")

print(f"AOI features : {len(aoi)}")
print(f"AOI CRS      : {aoi.crs}")

In [ ]:
# Clip SWORD to AOI
if aoi.crs != sword.crs:
    aoi_sword = aoi.to_crs(sword.crs)
else:
    aoi_sword = aoi

sword_clipped = sword.clip(aoi_sword)

print(f"SWORD global  : {len(sword):>8} reaches")
print(f"SWORD in AOI  : {len(sword_clipped):>8} reaches")

In [ ]:
# Clip additional datasets to AOI (loops automatically)
extra_clipped = {}

if not extra_datasets:
    print("No additional datasets to clip – skipping.")
else:
    for label, gdf in extra_datasets.items():
        aoi_matched = aoi.to_crs(gdf.crs) if aoi.crs != gdf.crs else aoi
        clipped = gdf.clip(aoi_matched)
        extra_clipped[label] = clipped
        print(f"{label}: {len(gdf)} → {len(clipped)} features after clip")

In [ ]:
# Quick visual check
fig, ax = plt.subplots(figsize=(12, 8))
sword_clipped.plot(ax=ax, linewidth=0.8, color="steelblue", label="SWORD")
aoi.to_crs(sword.crs).boundary.plot(ax=ax, color="red", linewidth=1.5,
                                     linestyle="--", label="AOI boundary")

# Plot additional datasets if any
colors = ["green", "orange", "purple", "brown"]
for i, (label, gdf) in enumerate(extra_clipped.items()):
    gdf.to_crs(sword.crs).plot(ax=ax, linewidth=0.8,
                                color=colors[i % len(colors)], label=label)

plt.title("Datasets clipped to AOI")
plt.legend()
plt.axis("off")
plt.show()

---
## 1.5 | Save outputs

In [ ]:
# Create output directories if they don't exist
os.makedirs(DATA_PROCESSED, exist_ok=True)
os.makedirs(DATA_OUTPUT, exist_ok=True)

# Save clipped SWORD
sword_clipped.to_file(OUT_SWORD_CLIPPED, driver="GPKG")
print(f"Saved SWORD clipped : {OUT_SWORD_CLIPPED}")

# Save additional clipped datasets
# Output filename = label in lowercase with underscores
for label, gdf in extra_clipped.items():
    safe_label = label.lower().replace(" ", "_")
    out = os.path.join(DATA_PROCESSED, f"{safe_label}_aoi_clip.gpkg")
    gdf.to_file(out, driver="GPKG")
    print(f"Saved {label:20s} : {out}")

---
## 1.6 | Interactive map

In [ ]:
# Interactive map of clipped SWORD – saved as HTML and opened in browser
m = sword_clipped.explore(
    column="strm_order",
    cmap="Blues",
    tooltip=[c for c in ["reach_id", "river_name", "strm_order", "slope", "width"]
             if c in sword_clipped.columns],
    tiles="CartoDB positron",
    legend=True
)

m.save(OUT_MAP)
webbrowser.open(OUT_MAP)
print(f"Map saved and opened: {OUT_MAP}")

---

---

In [ ]:
# # See available layers in the GeoPackage
# print(fiona.listlayers(GDW))
# # transforming GDW to vector data
# gdw = gpd.read_file(GDW)
# print(gdw.geometry.geom_type.unique())

# # Inspecting the data:
# print(gdw.head())
# print(gdw.columns)
# print(gdw.info())
# print(gdw.crs)

# #gdw.columns = gdw.columns.str.lower()
# print(gdw.columns)

In [ ]:
import sys, os
sys.path.append(os.path.join("..", "src"))
from _2_1_2_point import snap_gdw_to_sword_nodes

# Filter GDW to INSTREAM=1 first (Phase 1 from our plan)
gdw_filtered = gdw[gdw["INSTREAM"] == 1].copy()
print(f"GDW filtered (INSTREAM=1): {len(gdw_filtered)} / {len(gdw)}")

# Run the snap
matched, unmatched = snap_gdw_to_sword_nodes(gdw_filtered, naryn_nodes)

print("\nMatched:")
print(matched[["DAM_NAME", "node_id", "reach_id", "snap_dist", "n_buffer_steps"]])

print("\nUnmatched:")
if len(unmatched) > 0:
    print(unmatched[["DAM_NAME", "reason"]])
else:
    print("None")

## FFR

Available layers: ['FFR_river_network_v1']

Unique CONTINENT values: ['Africa', 'Asia', 'Australia', 'Europe', 'North America', 'South America']



In [ ]:

with fiona.open(IN_FFR, layer="FFR_river_network_v1") as src:
    print(f"Total features: {len(src)}")
    
# # Looking how many layers
# ffr_layers = fiona.listlayers(IN_FFR)
# print(f"Available layers: {ffr_layers}") # --> Only: FFR_river_network_v1

# with fiona.open(IN_FFR, layer="FFR_river_network_v1") as src:
#     continents = set()
#     for feature in src:
#         continents.add(feature["properties"]["CONTINENT"])

# print(f"Unique CONTINENT values: {sorted(continents, key=lambda x: (x is None, x))}")

In [ ]:
# continents = ['Africa', 'Asia', 'Australia', 'Europe', 'North America', 'South America']

# os.makedirs(DATA_PROCESSED, exist_ok=True)
# start = time.time()

# ffr_all = gpd.read_file(IN_FFR, layer="FFR_river_network_v1")

# elapsed = time.time() - start
# print(f"Loaded {len(ffr_all)} features in {elapsed/60:.1f} minutes")
# print(f"CRS: {ffr_all.crs}")

# # split by continent --> faster since everything is already in memory
# output_paths = {}
# for continent in tqdm(continents, desc="Splitting by continent"):
#     subset = ffr_all[ffr_all["CONTINENT"] == continent].copy()
#     clean_name = continent.lower().replace(" ", "_")
#     out_path = os.path.join(DATA_PROCESSED, f"ffr_{clean_name}.gpkg")
#     subset.to_file(out_path, driver="GPKG")
#     output_paths[continent] = out_path
#     tqdm.write(f"  {continent}: {len(subset)} features -> {out_path}")

# print("\nDone:")
# print(output_paths)

In [ ]:
print(ffr_all.columns.tolist())